# Scenario Suites

A **preset** describes a scenario's techniques, objectives, and parameters.
A **suite** is an ordered collection of preset references.
One launch runs those presets against a selected target through the ordinary scenario
execution path. The `scenario_suite` initializer registers a configurable catalog of
read-only suites and presets, similar to the target and converter initializers.

This notebook runs the text-only horizontal suite against the default `openai_chat`
target. Like the other scenario guides, it loads the shared Scanner configuration and
your configured target, adversarial, and scoring credentials.

In [ ]:
from pathlib import Path

from pyrit.registry import ScenarioPresetRegistry, ScenarioSuiteRegistry
from pyrit.setup import ConfigurationLoader
from pyrit.setup.initializers import ScenarioSuiteInitializer

config = ConfigurationLoader.from_yaml_file(Path("../../scanner/pyrit_conf.yaml"))
await config.initialize_pyrit_async()  # type: ignore

preset_registry = ScenarioPresetRegistry.get_registry_singleton()
suite_registry = ScenarioSuiteRegistry.get_registry_singleton()

Auto-discovered plaintext environment file ./.pyrit/.env will be loaded. Azure Key Vault through env_akv_ref is more secure for shared or deployed secrets; use .env.local only for deliberate local overrides. To inspect a resolved AKV-only configuration from a source checkout, run `python -m build_scripts.export_akv_environment`; it writes ~/.pyrit/.env_akv.


Found default environment files: ['./.pyrit/.env', './.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env
Loaded environment file: ./.pyrit/.env.local


Skipping scorer task_achieved_microsoft_decision: required target not found in TargetRegistry
Skipping scorer task_achieved_microsoft_decision_threshold_07: required target not found in TargetRegistry


## Configure the horizontal suite

`HorizontalTechniquesSuite` evaluates text with four families: inline jailbreak templates,
multilingual translation, encodings, and character obfuscation. All families run
the same objectives (by default the `airt_harms` dataset). Only the jailbreak
preset includes a baseline. Family-only presets are hidden from the individual Scanner list.
Translation uses complete objectives, and encoding/obfuscation use the disjoint
Foundry technique catalogs. Scoring follows the existing scenario defaults.

`dataset_names` can select other text-objective datasets. `dataset_fetch` defaults
to `if_missing`; `never` requires provisioned data. `always` is rejected because each
preset prepares the shared datasets on its own and would replace them once per family.
To refresh them, run the `refresh_datasets` initializer once before launch.

### Preset configuration

| Family | Explicit names or `["all"]` | Sample count |
| --- | --- | --- |
| Jailbreaks | `jailbreak_names` | `num_jailbreaks` |
| Languages | `languages` | `num_languages` |
| Encodings | `encoding_techniques` | `num_encodings` |
| Obfuscations | `obfuscation_techniques` | `num_obfuscations` |

Choose names or a count, not both. Names preserve order after normalization and
deduplication; `["all"]` expands the eligible catalog. Empty lists, unknown technique
names, mixed `all`/name lists, and invalid counts are rejected.
Automatic language selection excludes English; explicit languages may be outside the lexicon.

These examples use `selection_seed=42`. Each family has an independent stream, unaffected
by suite names, objective datasets, targets, or scorers. Catalog changes can
change samples, so use explicit lists to pin members.

In [ ]:
suite_args = {
    "num_jailbreaks": 5,
    "languages": ["French", "Spanish"],
    "encoding_techniques": ["all"],
    "num_obfuscations": 4,
    "selection_seed": 42,
}
initializer = ScenarioSuiteInitializer()
initializer.set_params_from_args(args={"suite_args": {"horizontal_techniques": suite_args}})
await initializer.initialize_async()  # type: ignore

suite = suite_registry.get_suite(name="horizontal_techniques").suite
for entry in suite.entries:
    preset = preset_registry.get_preset(name=entry.preset_name).preset
    print(f"{preset.name}: {preset.scenario_name}, techniques={preset.techniques}, baseline={preset.include_baseline}")

horizontal_techniques_jailbreaks: airt.jailbreak, techniques=['prompt_sending'], baseline=True
horizontal_techniques_multilingual: airt.multilingual, techniques=['prompt_sending'], baseline=False
horizontal_techniques_encoding: foundry.red_team_agent, techniques=['ascii_smuggler', 'atbash', 'base64', 'binary', 'caesar', 'morse', 'rot13', 'unicode_substitution', 'url'], baseline=False
horizontal_techniques_obfuscation: foundry.red_team_agent, techniques=['char_swap', 'character_space', 'flip', 'unicode_confusable'], baseline=False


## Configure the catalog from YAML

```yaml
initializers:
  - name: target
  - name: scorer
  - name: technique
  - name: scenario_suite
    args:
      tags:
        - default
      suite_args:
        horizontal_techniques:
          num_jailbreaks: 5
          languages: [French, Spanish]
          encoding_techniques: [all]
          num_obfuscations: 4
          selection_seed: 42
```

`tags` defaults to `["default"]`. Ordinary tags select their union in catalog order;
`["all"]` selects every entry, including untagged ones, and cannot be mixed with other
tags. Unknown and empty selections fail.

`suite_args` is a YAML/Python mapping keyed by selected catalog entry names. Overrides
replace constructor arguments; lists replace rather than append. `null` restores the
constructor default. For example, clear a catalog's explicit `num_jailbreaks` with `null`
before supplying `jailbreak_names`. Unknown arguments, even null-valued ones, and
overrides for unselected suites are errors. Counts use native numbers, not strings or
booleans. Nested GUI/CLI editing forms are not generated by this API.

## Run the suite

A launch request names the target and lists the reviewed entries, including stored-definition
versions, so changes since review can be detected. The suite execution service, which also
backs the GUI Scanner, preflights every entry before submitting any of them. It then queues
each preset as an ordinary scenario run against `openai_chat`, in order. Runs use the
configured default adversarial and scoring roles unless the request overrides them. Each
run fetches the shared objective dataset on demand and records the suite identity on its
scenario result. The initializer itself never executes attacks.

In [ ]:
import asyncio

from pyrit.backend.services import get_scenario_suite_execution_service
from pyrit.models import (
    LaunchScenarioSuiteRequest,
    ScenarioSuiteExecutionStatus,
    ScenarioSuiteLaunchEntrySnapshot,
)

request = LaunchScenarioSuiteRequest(
    target_name="openai_chat",
    entries=[
        ScenarioSuiteLaunchEntrySnapshot(
            entry_id=entry.entry_id, preset_name=entry.preset_name, origin="registered", preset_version=None
        )
        for entry in suite.entries
    ],
)
suite_service = get_scenario_suite_execution_service()
execution = await suite_service.launch_async(name=suite.name, request=request)  # type: ignore

while execution.status in (ScenarioSuiteExecutionStatus.QUEUING, ScenarioSuiteExecutionStatus.IN_PROGRESS):
    await asyncio.sleep(10)  # type: ignore
    execution = await suite_service.get_execution_async(execution_id=execution.suite_execution_id)  # type: ignore

print(f"Suite execution: {execution.status.value}")
for entry in execution.entries:
    print(f"{entry.entry_position}. {entry.preset_name}: {entry.state.value}")

TextAdaptive: _EXCLUDED_TECHNIQUES entries ['prompt_sending'] are not in the current scenario-techniques catalog ['best_of_n', 'bijection', 'char_split', 'code_attack', 'code_attack_framed', 'context_compliance', 'context_wrap', 'crescendo', 'crescendo_history_lecture', 'crescendo_journalist_interview', 'crescendo_movie_director', 'crescendo_simulated', 'flip', 'goat', 'many_shot', 'pair', 'red_teaming', 'role_play_movie_script', 'role_play_persuasion', 'role_play_persuasion_written', 'role_play_trivia_game', 'role_play_video_game', 'skeleton_key', 'split_payload', 'tap', 'violent_durian']; the exclusion is a no-op for those entries. Remove stale entries or update the catalog.
C:\Source\PyRIT-internal\PyRIT\pyrit\backend\models\scorers.py:36: UserWarning: Field name "register" in "CreateScorerRequest" shadows an attribute in parent "BaseModel"
  class CreateScorerRequest(BaseModel):
C:\Source\PyRIT-internal\PyRIT\pyrit\backend\models\converters.py:93: UserWarning: Field name "register"

Executing Jailbreak:   0%|          | 0/6 [00:00<?, ?attack/s]

Executing Multilingual:   0%|          | 0/2 [00:00<?, ?attack/s]

Executing RedTeamAgent:   0%|          | 0/9 [00:00<?, ?attack/s]

Executing RedTeamAgent:   0%|          | 0/4 [00:00<?, ?attack/s]

Suite execution: COMPLETED
1. horizontal_techniques_jailbreaks: COMPLETED
2. horizontal_techniques_multilingual: COMPLETED
3. horizontal_techniques_encoding: COMPLETED
4. horizontal_techniques_obfuscation: COMPLETED


## Summarize the results

Each row counts the latest outcome for each technique/objective execution unit.
This summary leaves raw conversations in memory rather than printing them here.

In [ ]:
from pyrit.analytics import compute_scenario_statistics
from pyrit.backend.services import get_scenario_run_service

run_service = get_scenario_run_service()
results = {
    entry.preset_name: await run_service.get_run_results_async(scenario_result_id=entry.run.scenario_result_id)  # type: ignore
    for entry in execution.entries
    if entry.run is not None
}

for preset_name, result in results.items():
    counts = compute_scenario_statistics(result).overall
    outcomes = counts.outcomes
    assert outcomes is not None
    print(
        f"{preset_name}: {counts.completed}/{counts.planned} completed, "
        f"{outcomes.successes} succeeded, {outcomes.failures} failed, "
        f"{outcomes.errors} errors, {outcomes.undetermined} undetermined"
    )

horizontal_techniques_jailbreaks: 24/24 completed, 0 succeeded, 24 failed, 0 errors, 0 undetermined
horizontal_techniques_multilingual: 8/8 completed, 0 succeeded, 8 failed, 0 errors, 0 undetermined
horizontal_techniques_encoding: 36/36 completed, 0 succeeded, 36 failed, 0 errors, 0 undetermined
horizontal_techniques_obfuscation: 16/16 completed, 1 succeeded, 15 failed, 0 errors, 0 undetermined
